# Exercise: Transition State Optimization

<div style="border-radius:2px; overflow:hidden; border:1px solid #dee2e6;">
  <div style="font-weight:600; padding:10px 14px; border-bottom:1px solid #bee5eb; background-color:#d1ecf1; color:#0c5460;">Learning objectives</div>
  <div style="padding:14px;"><p>Use a constrained approach scan to find an SN2 transition-state guess.</p><p>Refine the guess with Sella and inspect the optimized saddle.</p><p>Check the unstable vibrational mode and, optionally, trace both reaction-path directions.</p></div>
</div>

**Model choice.** [AIMNet2-2025](https://github.com/isayevlab/aimnetcentral) supports the elements and molecular charge needed for this anionic reaction. Its molecular energies and forces drive the scan and saddle search; the transition state must still be checked.

Complete the `...` placeholders in **Your turn** cells. Other cells are provided.


**Kernel:** AIMNet2, with Sella installed. Use a fresh kernel.

**Learning goals:** construct a transition-state guess from a constrained scan; refine a saddle;
check the unstable vibrational mode; optionally follow both reaction-path directions.

![SN2 reaction](SN2.png)

We model **Br- + CH3Cl → CH3Br + Cl-**, with total charge **-1**. This is a gas-phase model
calculation. Solvent effects and free-energy corrections are outside this exercise.

**Predict:** which bond forms and which bond breaks? What motion should the unstable mode show?

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
sys.path.insert(0,str(Path.cwd().parent))


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from demo_tools.view_mols import view

from demo_tools.helpers import start_exercise
from demo_tools.optimization import optimize_ase

_, OUTPUT = start_exercise()

Python: /opt/conda/envs/aimnet2/bin/python
Inputs: /scratch/user/nchopper/mlip_demo_update/demo/TransitionStates
New results: /scratch/user/nchopper/mlip_demo_update/demo/TransitionStates/results/20261101-205131-5e6d3d


## 1. Check the input
The six-atom XYZ uses C=0, Cl=1, Br=5. The neutral setting from the original demonstration has been
changed to -1 for the stated anionic reaction. This changes the potential-energy surface and requires
fresh calculations; never reuse the original vibration cache.


In [3]:
'''
C        0.00000000       0.00000000       0.00000000
Cl       1.78400000       0.00000000       0.00000000
H       -0.34400000       1.02900000      -0.00000000
H       -0.34400000      -0.51500000      -0.89100000
H       -0.34400000      -0.51500000       0.89100000
Br      -3.00000000       0.00000000       0.00000000
'''


molecule = read('sn2_initial_structure.xyz')
view(molecule)  # Visualize the molecule after reading it from the file


<div style="border-radius:2px; overflow:hidden; border:1px solid #dee2e6;">
  <div style="font-weight:600; padding:10px 14px; border-bottom:1px solid #bee5eb; background-color:#d1ecf1; color:#0c5460;">From scan to saddle guess</div>
  <div style="padding:14px;">The scan moves Br toward carbon and relaxes the remaining allowed coordinates with AIMNet2. An interior energy maximum supplies a starting guess for <code>Sella</code>; it is not yet a validated transition state.</div>
</div>

## 2. Scan the approaching C-Br distance
Fix C and Br at each separation and relax the other atoms. This is a constrained path used to find
a guess, not proof of a minimum-energy reaction path. A scan may fail to show an interior maximum;
if so, inspect the geometries and adjust the scan instead of selecting an endpoint automatically.


**Your turn.** Set the total charge for Br- + CH3Cl. The model and scan constraints are supplied.


In [4]:
# We need to get an initial guess for a transition state structure.
#  This can be done by manipulating the positions of the atoms in the molecule.
#  For example, we can move the nucleophile closer to the carbon atom that is being attacked.
from aimnet.calculators import AIMNet2ASE
calculator = AIMNet2ASE(
    'aimnet2-2025',  # Specify the AIMNet model to use
    charge = ...  # Specify the charge of the molecule
)

# Br - C - Cl is currently aligned along x.  We want to fix these atoms atoms along this line
#  We will also fix the position of the carbon entirely
#  Carbon index = 0
#  Chlorine index = 1
#  Bromine index = 5
from ase.constraints import FixedLine, FixAtoms, FixInternals
cbr_bond_indices = [0,5]
cbr_constraint = FixInternals(bonds=[[3.0, cbr_bond_indices]])
molecule.set_constraint(cbr_constraint)


Warp CUDA warning: Could not find or load the NVIDIA CUDA driver. GPU execution will not be available.


ConnectionError: HTTPSConnectionPool(host='storage.googleapis.com', port=443): Max retries exceeded with url: /aimnetcentral/aimnet2v2/AIMNet2/aimnet2_2025_b973c_d3_0.pt (Caused by NewConnectionError("HTTPSConnection(host='storage.googleapis.com', port=443): Failed to establish a new connection: [Errno 101] Network is unreachable"))

In [ ]:
# Attach calculator and move Br closer to C iteratively
from ase.optimize import BFGS
molecule.calc = calculator

res = []
for cbr_distance in np.linspace(3.0, 1.7, 14):
    # Reduce the C-Br distance by 0.1 Å in each step
    molecule.set_distance(0, 5, cbr_distance, fix=0)
    molecule.set_constraint()
    constraint = FixAtoms(indices=[0, 5])
    molecule.set_constraint(constraint)
    dyn = optimize_ase(molecule, optimizer=BFGS, fmax=0.01)
    res.append((molecule.get_distance(0, 5), molecule.get_potential_energy(), molecule.copy()))





In [ ]:
import matplotlib.pyplot as plt
distances = [r[0] for r in res]
energies = [r[1] - res[0][1] for r in res]
plt.plot(distances, energies)
plt.xlabel('C-Br Distance (Å)')
plt.ylabel('Potential Energy (eV)')
plt.title('SN2 Reaction Energy Profile')
# plt.show()

for i,r in enumerate(res):
    plt.text(distances[i]-0.01, energies[i]+0.01, f'{i}', fontsize=8)

<div role="alert" class="alert alert-warning">Select an interior maximum only after looking at the scan. A boundary maximum or a monotonic scan needs a different starting guess.</div>

In [ ]:
def get_ts_index(energies):
    # If i-1 < i < i+1, then i is a local maximum
    for i in range(1, len(energies)-1):
        if energies[i-1] < energies[i] and energies[i] > energies[i+1]:
            return i
    return None

ts_index = get_ts_index(energies)
assert ts_index is not None, 'No interior maximum: inspect and adjust the scan'
ts_guess = res[ts_index]
ts_guess_structure = ts_guess[2]


## 3. Refine an interior maximum with Sella
The first interior local maximum is used as a starting guess. Remove the scan constraints before
saddle optimization. Convergence of this optimizer is necessary, but does not establish that the
saddle connects the intended reactant and product.


**Your turn.** Attach the calculator, create Sella with internal=True, and run it with fmax=1e-4 and at most 1000 steps.


In [ ]:
from sella import Sella
ts_guess_structure.set_constraint()
ts_guess_structure.calc = ...
opt = ...  # TODO: create the saddle optimizer
...  # TODO: run the saddle optimization

## 4. Validate the saddle using vibrations
For an unconstrained nonlinear molecule, expect one chemically meaningful imaginary frequency at
a first-order saddle, plus near-zero translation/rotation modes. Small imaginary frequencies can be
numerical artifacts
Inspect the displacement: does it form C-Br while breaking C-Cl?

This finite-difference calculation makes multiple force evaluations. Its cache lives in this run's results directory


In [ ]:
# Verify the transition state by checking vibrational frequencies
from ase.vibrations import Vibrations
vib = Vibrations(ts_guess_structure, name=str(OUTPUT/'vib'))
vib.run()
vib.summary()

In [ ]:
frequencies = vib.get_frequencies()
n_imaginary = int(np.count_nonzero(np.abs(np.imag(frequencies)) > 1e-6))

<div role="alert" class="alert alert-info">There should be one (and only one) imaginary vibrational mode. </div>

In [ ]:
vib.write_mode(0)
mode0 = read(OUTPUT/'vib.0.traj', index=':')
view(mode0, animate=True)


## 5. Optional: follow both IRC branches
Enable only after inspecting the saddle and unstable mode. Each direction starts from a fresh copy
of the saddle and writes to its own trajectory; we do not infer a branch boundary from an energy jump.
Forward and reverse labels do not inherently identify reactants and products. Inspect both endpoints.
This extension can take substantially longer than the scan.


<div style="border-radius:2px; overflow:hidden; border:1px solid #dee2e6;">
  <div style="font-weight:600; padding:10px 14px; border-bottom:1px solid #bee5eb; background-color:#d1ecf1; color:#0c5460;">Optional IRC follow-up</div>
  <div style="padding:14px;">Run forward and reverse IRC branches from fresh copies of the validated saddle. Inspect their endpoints to identify reactant and product sides.</div>
</div>

In [ ]:
from sella import IRC
for direction in ['forward', 'reverse']:
    branch = ts_guess_structure.copy()
    branch.calc = calculator
    opt = IRC(branch, trajectory=str(OUTPUT/f'irc_{direction}.traj'), dx=0.1, eta=1e-4, gamma=0.4)
    opt.run(fmax=0.1, steps=1000, direction=direction)


In [ ]:
forward = read(OUTPUT/'irc_forward.traj', index=':')
reverse = read(OUTPUT/'irc_reverse.traj', index=':')
irc = reverse[::-1] + forward[1:]
view(irc, animate=True, bonds='recompute')


In [ ]:
energies = [irc[i].get_potential_energy() for i in range(len(irc))]
import matplotlib.pyplot as plt
plt.plot(energies - np.min(energies))
plt.xlabel('IRC Step')
plt.ylabel('Potential Energy')
plt.show()

## Try, explain, and report
1. Does the unstable mode describe the expected bond exchange?
2. If you ran IRC, do the endpoints correspond to the intended reactant and product sides?



# Summary

<div style="border-radius:2px; overflow:hidden; border:1px solid #dee2e6;">
  <div style="font-weight:600; padding:10px 14px; border-bottom:1px solid #c3e6cb; background-color:#d4edda; color:#155724;">What you learned</div>
  <div style="padding:14px;"><p>You used a scan to seed a saddle-point optimization.</p><p>A transition-state assignment is supported by its unstable mode and, when run, the reaction paths on either side.</p></div>
</div>